# Optional: LLM Agent DemoRun Claude on a skinned diagnosis task. **Requires ANTHROPIC_API_KEY in .env file.**

In [ ]:
import sys, ossys.path.insert(0, os.path.join(os.getcwd(), "demos"))# Load .envenv_path = os.path.join(os.getcwd(), ".env")if os.path.exists(env_path):    with open(env_path) as f:        for line in f:            line = line.strip()            if line and not line.startswith("#") and "=" in line:                key, _, value = line.partition("=")                os.environ.setdefault(key.strip(), value.strip())API_KEY = os.environ.get("ANTHROPIC_API_KEY")if not API_KEY:    print("ANTHROPIC_API_KEY not found. Set it in .env to run this demo.")else:    print("API key found.")

In [ ]:
if API_KEY:    import anthropic    from _shared import make_disease_system, oracle_agent, random_agent    from alienbio.bio import AgentInterface, DiagnoseTask    from alienbio.scenarios.skinning import generate_name_map, generate_description    system, baseline, perturbs = make_disease_system(seed=42)    name_map = generate_name_map(system, seed=42)    desc = generate_description(system, detail_level=2, name_map=name_map, seed=42)    task = DiagnoseTask(perturbs[:4], applied_index=0)    iface = AgentInterface(system)    candidate_descs = []    for i, p in enumerate(task.candidates):        skinned = name_map.get(p.target_reaction, p.target_reaction)        candidate_descs.append(f"  {i}: {p.kind} affecting {skinned}")    prompt = f"""You are diagnosing an alien biological system.{desc}The system is diseased. Which perturbation (0-{len(task.candidates)-1}) was applied?{chr(10).join(candidate_descs)}Reply with ONLY the number."""    client = anthropic.Anthropic(api_key=API_KEY)    response = client.messages.create(model="claude-sonnet-4-5-20250929", max_tokens=10,        messages=[{"role": "user", "content": prompt}])    llm_answer = response.content[0].text.strip()    try:        llm_pred = int(llm_answer)    except ValueError:        llm_pred = 0    print(f"Claude: predicted={llm_pred}, score={task.score(iface, llm_pred).score:.2f}")    print(f"Oracle: score={task.score(iface, oracle_agent(iface, task)).score:.2f}")    print(f"Random: score={task.score(iface, random_agent(iface, task)).score:.2f}")